In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
from string import Template
from IPython.display import Markdown, display
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
import llm_tools as llt


from matplotlib import pyplot as plt
from utils import parse_casenum, weighted_mean, extract_json

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)


In [2]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 8000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 4000
REQUESTS_PER_BATCH = 1000
REPLACE = False

In [3]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)


In [4]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))

In [5]:
# Flat df of cases with lists of hearing dates and hearing results

flat_df = {}
df['canonical_casenum'] = df['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])
for idx, row in df.iterrows():
    date = pd.to_datetime(row['date'])
    project_result = row["project_result"]
    canonical_casenum = row['canonical_casenum']
    casenum = row['title']
    if canonical_casenum not in flat_df:
        flat_df[canonical_casenum] = {
            "hearing_dates": [],
            "project_results": [],
            "casenums": [],
            "agenda_texts": [],
            "minutes_summaries": []
        }
    flat_df[canonical_casenum]["hearing_dates"].append(date)
    flat_df[canonical_casenum]["project_results"].append(project_result)
    flat_df[canonical_casenum]["casenums"].append(casenum)
    flat_df[canonical_casenum]["agenda_texts"].append(row["agenda_text"])
    flat_df[canonical_casenum]["minutes_summaries"].append(row["minutes_summary"])

flat_df = pd.DataFrame(flat_df, index=['hearing_dates', 'project_results', 'casenums', 'agenda_texts', 'minutes_summaries']).T.reset_index().rename(columns={'index': 'canonical_casenum'})

In [6]:
# Calculate average interval between hearings for cases with multiple hearings

flat_df['n_hearings'] = flat_df['hearing_dates'].apply(len)
flat_df['last_result'] = flat_df['project_results'].apply(lambda x: x[-1])
flat_df['first_result'] = flat_df['project_results'].apply(lambda x: x[0])
flat_df["average_hearing_interval"] = flat_df['hearing_dates'].apply(lambda x: np.mean(np.diff(sorted(x)).astype('timedelta64[D]')) if len(x) > 1 else np.nan)
mask = (flat_df["n_hearings"]>1) & (flat_df["first_result"]=="DELAYED") 
flat_df.loc[mask, "average_hearing_interval"].mean()

Timedelta('56 days 02:10:54.545454545')

In [7]:
# Calculate statistics for delayed cases

NumDelayedCases = (flat_df["first_result"]=="DELAYED").sum()

mask = (flat_df["n_hearings"]>1) & (flat_df["first_result"]=="DELAYED")
PctHeardAgain = mask.sum() / NumDelayedCases
TimeBetweenHearings = weighted_mean(
    (flat_df.loc[mask, "average_hearing_interval"]/pd.to_timedelta(1, unit='D')),
    (flat_df.loc[mask, "n_hearings"]-1)
)
PctUltimatelyApproved = ((flat_df["last_result"]=="APPROVED") & (flat_df["first_result"]=="DELAYED")).sum() / NumDelayedCases
PctUltApprovedwMods = ((flat_df["last_result"].isin(["APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS", "APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS"])) & (flat_df["first_result"]=="DELAYED")).sum() / NumDelayedCases

RESULTS = {
    "NumDelayedCases": f"{NumDelayedCases:.0f}",
    "PctHeardAgain": f"{PctHeardAgain*100:.1f}",
    "TimeBetweenHearings": f"{TimeBetweenHearings:.0f}",
    "PctUltimatelyApproved": f"{PctUltimatelyApproved*100:.1f}",
    "PctUltApprovedwMods": f"{PctUltApprovedwMods*100:.1f}"
}

RESULTS


{'NumDelayedCases': '71',
 'PctHeardAgain': '93.0',
 'TimeBetweenHearings': '54',
 'PctUltimatelyApproved': '29.6',
 'PctUltApprovedwMods': '56.3'}

In [8]:
# Summarize changes for delayed cases that ultimately get approved

mask = (flat_df["first_result"]=="DELAYED") & (flat_df["last_result"]!="DELAYED")

PROMPT = Template("""
In what follows, I will provide you initial and final hearing information for a single Los Angeles City Planning Commission case. For the initial hearing, the initial agenda and a summary of the outcome are provided. For the final hearing, the final agenda and a summary of the outcome are provided.  Return a JSON object with the following structure:

{
    "summary_of_changes": <A summary of the changes between the initial and final hearings, highlighting what was added, removed, or modified in proposed project. If none, indicate that there were no changes.>,
    "degree_of_changes": <A description of the degree of change between the initial project proposal and the ultimately approved outcome. Your options are: "NONE", "MINOR", "MAJOR".>
}

<initial_agenda>
$initial_agenda
</initial_agenda>

<initial_outcome>
$initial_outcome
</initial_outcome>

<final_agenda>
$final_agenda
</final_agenda>

<final_outcome>
$final_outcome
</final_outcome>
""")

for idx, row in flat_df.loc[mask].iterrows():
    initial_agenda = row["agenda_texts"][0]
    final_agenda = row["agenda_texts"][-1]
    initial_outcome = row["minutes_summaries"][0]
    final_outcome = row["minutes_summaries"][-1]
    prompt = PROMPT.substitute(
        initial_agenda=initial_agenda,
        final_agenda=final_agenda,
        initial_outcome=initial_outcome,
        final_outcome=final_outcome
    )
    response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    j = extract_json(response["response"])
    flat_df.at[idx, "summary_of_changes"] = j["summary_of_changes"]
    flat_df.at[idx, "degree_of_changes"] = j["degree_of_changes"]

flat_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "delay_impacts_flat_df.parquet"))


In [26]:
row = flat_df.loc[mask].sample(1).iloc[0]

print(row["hearing_dates"])
print(row["canonical_casenum"])
print(row["project_results"])
print("")
print(row["degree_of_changes"])
print(row["summary_of_changes"])

[Timestamp('2020-03-12 00:00:00'), Timestamp('2020-04-23 00:00:00')]
CPC-2019-4908
['DELAYED', 'APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS']

NONE
There were no changes to the proposed project between the initial and final hearings. The project description, requested actions, and all specifications (102 dwelling units, 12 Very Low Income units, 45-foot-5-inch height, 2.65:1 FAR, open space reduction, rear yard setback reduction, and height waiver) remained identical. The initial hearing resulted in a continuance to a later date rather than a decision on the merits. At the final hearing, the project was approved unanimously 9-0 with Conditions of Approval as modified by the Commission and a Staff Technical Modification dated April 23, 2020, but the underlying project proposal itself was unchanged.


In [27]:
flat_df["degree_of_changes"].value_counts()

degree_of_changes
MINOR    33
NONE     25
MAJOR     8
Name: count, dtype: int64